In [ ]:
# SBERT 모델을 사용하기 위해 라이브러리 설치 
# !pip install sentence-transformers

### SBERT
- BERT모델 
    - 문장 이해용 Encoder
    - 문장의 쌍을 비교 
- SBERT모델 
    - 문장 의미를 임베딩 
    - 벡터의 비교용

In [2]:
import torch 
from sentence_transformers import SentenceTransformer, util

In [4]:
# 다목적 한국 SBERT 
model_name = 'jhgan/ko-sroberta-multitask'
# 문장 유사도 특화 모델 
model_name2 = 'BM-K/KoSimCSE-roberta-multitask'

sbert = SentenceTransformer(model_name)
sbert2 = SentenceTransformer(model_name2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9874.67it/s]


In [5]:
# 각 모델의 설정 세팅(최대 길이 설정)
sbert.max_seq_length = 256
sbert2.max_seq_length = 256

In [6]:
doc1 = "이 카메라는 색감이 자연스럽고 배터리도 오래간다"
doc2 = "배터리 성능이 좋고 사진 품질이 뛰어나다"

In [7]:
# 추론 모드 
with torch.inference_mode():
    emb1 = sbert.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 두 벡터간의 유사도를 확인 
cos_sim = util.cos_sim(emb1, emb2).item()
round(cos_sim, 4)

0.6948

In [8]:
# 추론 모드 
with torch.inference_mode():
    emb1 = sbert2.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert2.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 두 벡터간의 유사도를 확인 
cos_sim = util.cos_sim(emb1, emb2).item()
round(cos_sim, 4)

0.734

In [9]:
sentences = [
    '하이닉스 주가가 올랐다', 
    '코스피가 상승 마감했다', 
    '날씨가 안 좋아서 항공편이 취소됬다'
]

with torch.inference_mode():
    embs = sbert2.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)

embs

tensor([[-0.0174, -0.0515,  0.0087,  ..., -0.0166,  0.0082,  0.0246],
        [ 0.0062, -0.0016,  0.0583,  ..., -0.0206, -0.0037,  0.0356],
        [-0.0446,  0.0469,  0.0770,  ...,  0.0148, -0.0263, -0.0369]])

In [12]:
# embs -> encode() 함수의 결과값 ( CLS + 단어 벡터들의 평균 값 )
sim_metrix = util.cos_sim(embs, embs)
sim_metrix

tensor([[1.0000, 0.3316, 0.0309],
        [0.3316, 1.0000, 0.1530],
        [0.0309, 0.1530, 1.0000]])

In [20]:
new_sentence = "증시가 강세였다"

new_emb = sbert2.encode(new_sentence, convert_to_tensor=True, normalize_embeddings=True)

# 유사도가 높은 상위의 2개를 선택 
hits = torch.topk(util.cos_sim(new_emb, embs).squeeze(0), k =2 )

In [24]:
hits

torch.return_types.topk(
values=tensor([0.6014, 0.4809]),
indices=tensor([1, 0]))

In [26]:
for score, idx in zip( hits.values.tolist(), hits.indices.tolist() ):
    # score : 코사인 유사도
    # idx : 위치
    print(f"유사 문장 : {sentences[idx]}, 유사도 : {round(score, 3)}")

유사 문장 : 코스피가 상승 마감했다, 유사도 : 0.601
유사 문장 : 하이닉스 주가가 올랐다, 유사도 : 0.481


### 실습 
- ratings_train.txt 파일 로드 
- 데이터 튜닝 (결측치 제거, 정규화, 중복 데이터 제거, 글자수 1자리 이하 제거)
- train, test 데이터 분할 
- sbert 모델은 'jhgan/ko-sroberta-multitask' 사용
- Dataset 선언 
    - 입력 받는 데이터는 텍스트, 라벨 
    - 생성자 함수 
        - 입력 받은 텍스트(sentences)들을 sbert 모델을 이용하여 임베딩 
        - 라벨 데이터는 self변수로 저장 
    - `__len__()`
        - 길이를 되돌려준다. 
    - `__getitem__()`
        - 임베딩된 데이터에서 특정 위치의 데이터와 같은 위치의 라벨 데이터를 되돌려준다. 
- DataLoader를 통해서 batch data를 생성을 설정 
- 학습 모델을 생성 
    - ML
        - SVC 모델  
    - DL
        - 비선형 활성화 함수를 포함한 다중 구조로 생성 
- 예측값을 이용하여 평가 지표 확인 

In [27]:
import re 
import pandas as pd 
import torch
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.svm import SVC

In [28]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')

In [30]:
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()
    return text

In [32]:
# 결측치 제거
df.dropna(inplace=True)
# 텍스트 정규화
df['document'] = df['document'].map(normalize)
# 중복 데이터 제거
df.drop_duplicates('document', inplace=True)
# 길이가 1이하인 데이터 제거
df = df.loc[ df['document'].str.len() > 1,  ]

In [33]:
len(df)

144637

In [34]:
# 랜덤한 5000개의 데이터를 추출 
df2 = df.sample(n = 5000, random_state=42).reset_index(drop=True)
df2['label'].value_counts()

label
1    2520
0    2480
Name: count, dtype: int64

In [35]:
# train, test 데이터로 분할 
train_df, test_df = train_test_split(
    df2, test_size=0.2, random_state=42, stratify=df2['label']
)

In [36]:
# 모델 선택 
sbert3 = SentenceTransformer('jhgan/ko-sroberta-multitask')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9684.48it/s]


In [37]:
# Dataset 선언 
class SBERTDataset(Dataset):
    # 생성자 함수 -> document, label 데이터를 입력 받는다. 
    def __init__(self, document, labels):
        # 생성자 함수에서 document 데이터를 임베딩 
        self.labels = torch.tensor(labels, dtype=torch.long)

        # 모델의 추론모드 사용
        with torch.inference_mode():
            self.emb = sbert3.encode(
                document, convert_to_tensor=True, normalize_embeddings=True
            )
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]

In [ ]:
class SBERTDataset2(Dataset):
    # 생성자 함수 -> document, label 데이터를 입력 받는다. 
    def __init__(self, document, labels):
        # 생성자 함수에서 document 데이터를 임베딩 
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.document = document
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        with torch.inference_mode():
            emb = sbert3.encode(
                self.document[idx], convert_to_tensor=True, normalize_embeddings=True
            )       # 길이가 768인 1차원 텐서 
        return emb, self.labels[idx]

In [41]:
train_ds = SBERTDataset(train_df['document'].tolist(), train_df['label'].tolist())

In [42]:
test_ds = SBERTDataset(test_df['document'].tolist(), test_df['label'].tolist())

In [62]:
train_ds2 = SBERTDataset2(train_df['document'].tolist(), train_df['label'].tolist())
test_ds2 = SBERTDataset2(test_df['document'].tolist(), test_df['label'].tolist())

In [46]:
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128, shuffle=True)

In [63]:
train_dl2 = DataLoader(train_ds2, batch_size=128, shuffle=True)
test_dl2 = DataLoader(test_ds2, batch_size=128, shuffle=True)

In [48]:
# 다중 퍼셉트론 구조 딥러닝 모델을 생성 
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim = 256, num_classes = 2, dropout = 0.5):
        super().__init__()

        # 다중 퍼셉트론층 구성 
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), 
            # 비선형 구조를 이해하기 위한 비선형 활성화 함수 
            nn.ReLU(), 
            # 과적합 방지용
            nn.Dropout(dropout), 
            nn.Linear(hidden_dim, num_classes)
        )
    # 순전파 함수 생성 -> 독립 변수의 값을 받아온다 
    def forward(self, X):
        # X : 독립 변수 ( SBERT 모델을 통해서 임베딩 데이터를 배치로 묶은 데이터 )
        result = self.net(X)
        return result

In [49]:
# sbert 모델의 출력의 차원의 수를 확인 
in_dim = sbert3.get_embedding_dimension()
in_dim

768

In [70]:
clf = MLPModel(in_dim)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(clf.parameters(), lr = 2e-04)

In [71]:
clf.train()

for epoch in range(5):
    total = 0.0

    for X, y in train_dl:
        # X : 임베딩 데이터들
        # y : label들 

        optimizer.zero_grad()
        logits = clf(X)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total += loss.item() * X.size(0)
    print(f"epoch : {epoch}, loss : {round(total / len(train_ds), 4)}")

epoch : 0, loss : 0.6715
epoch : 1, loss : 0.6044
epoch : 2, loss : 0.522
epoch : 3, loss : 0.4619
epoch : 4, loss : 0.4319


In [74]:
# 예측 
clf.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for X, y in test_dl:
        logits = clf(X)
        pred = logits.argmax(dim = -1).tolist()

        y_true += y.tolist()
        y_pred += pred
    
print(y_true)
print(y_pred)

[1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 

In [75]:
print(classification_report(y_pred, y_true))

              precision    recall  f1-score   support

           0       0.84      0.83      0.83       505
           1       0.83      0.84      0.83       495

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83      1000
weighted avg       0.83      0.83      0.83      1000



In [83]:
# Dataset에서 데이터를 가져온다. -> 임베딩데이터, 라벨 데이터 
X_train, y_train = train_ds[0:len(train_ds)]

In [88]:
import numpy as np

In [90]:
X_train = np.array(X_train)
y_tain = np.array(y_train)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_7340\35555528.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  X_train = np.array(X_train)
C:\Users\ekfla\AppData\Local\Temp\ipykernel_7340\35555528.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_tain = np.array(y_train)


In [91]:
X_test, y_test = test_ds[0:len(test_ds)]
X_test = np.array(X_test)
y_test = np.array(y_test)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_7340\377969095.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  X_test = np.array(X_test)
C:\Users\ekfla\AppData\Local\Temp\ipykernel_7340\377969095.py:3: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_test = np.array(y_test)


In [99]:
# SVC 모델을 생성 
svc = SVC(random_state=42)

In [ ]:
svc.fit(X_train, y_train)

In [101]:
# 예측값 생성 
pred = svc.predict(X_test)

In [102]:
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

           0       0.86      0.84      0.85       509
           1       0.84      0.86      0.85       491

    accuracy                           0.85      1000
   macro avg       0.85      0.85      0.85      1000
weighted avg       0.85      0.85      0.85      1000

